# 05 Simple Contrastive Learning Pretraining

This notebook prepares prefix-based sequence samples before the initial self-supervised contrastive learning experiment.

The purpose of this experiment is to construct training batches for a simple contrastive learning setup, where each vehicle sequence is converted into one or more prefix samples and two augmented views are generated for each sample.

This experiment represents an early stage in the development of the proposed semi-supervised survival modelling framework. It is kept in the repository for transparency and to show how the final approach evolved.

## Inputs

- prepared dictionaries of full vehicle sequences from the sequence preparation notebook;
- transformation utilities for prefix generation and augmentation.
- utilities for dataset preparation and dataloader objects before training

## Outputs

- pretrained encoder for downstream survival modelling fine-tuning

In [ ]:
# Set project paths and enable imports from src and models directories

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "processed" / "sequences"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [SRC_DIR, DATA_DIR, MODELS_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

print(f"Notebook location: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Sequences directory: {DATA_DIR}")

In [ ]:
# check runtime GPU
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# load data
import pickle

with open(DATA_DIR / "train_sequences_full.pkl", "rb") as f:
    train_seq = pickle.load(f)

with open(DATA_DIR / "val_sequences_full.pkl", "rb") as f:
    val_seq = pickle.load(f)

print(len(train_seq), len(val_seq))

In [ ]:
# Inspect the first loaded sequence
first_vehicle = next(iter(train_seq))
sample_seq = train_seq[first_vehicle]

print("Vehicle id:", first_vehicle)
print("Keys:", sample_seq.keys())
print("Numerical shape:", sample_seq["numerical_sequence"].shape)
print("Histogram shape:", sample_seq["histogram_sequence"].shape)
print("Sequence length:", sample_seq["sequence_length"])

In [ ]:
from sequence_preprocessing import build_prefixes_for_all_sequences
from ssl_dataset import SSLPrefixDataset, ssl_prefix_collate_fn

print("Imports successful.")

In [ ]:
# build prefixes from the loaded training sequences
train_prefixes = build_prefixes_for_all_sequences(
    sequence_dict=train_seq,
    proportions=(0.3, 0.6, 0.9),
    min_prefix_len=5,
    strict_prefix=True,
)

print("Number of prefixes:", len(train_prefixes))

In [ ]:
# inspect one prefix
first_prefix_id = next(iter(train_prefixes))
first_prefix = train_prefixes[first_prefix_id]

print("Prefix id:", first_prefix_id)
print("Keys:", first_prefix.keys())
print("Prefix length:", first_prefix["prefix_length"])
print("Numerical shape:", first_prefix["numerical_sequence"].shape)
print("Histogram shape:", first_prefix["histogram_sequence"].shape)

In [ ]:
# create PyTorch dataset
ssl_dataset = SSLPrefixDataset(
    prefix_dict=train_prefixes,
    numerical_mask_probability=0.3,
    numerical_noise_std=0.5,
    numerical_dropout_probability=0.3,
    histogram_mask_probability=0.4,
    random_seed=42,
)

print("Dataset size:", len(ssl_dataset))

In [ ]:
# inspect one sample
sample = ssl_dataset[0]

print("Sample keys:", sample.keys())
print("Prefix id:", sample["prefix_id"])
print("Sequence length:", sample["sequence_length"])
print("view_1_numerical shape:", sample["view_1_numerical"].shape)
print("view_1_histogram shape:", sample["view_1_histogram"].shape)
print("view_2_numerical shape:", sample["view_2_numerical"].shape)
print("view_2_histogram shape:", sample["view_2_histogram"].shape)

In [ ]:
# create the dataloader
from torch.utils.data import DataLoader

ssl_loader = DataLoader(
    ssl_dataset,
    batch_size=128,
    shuffle=True,
    collate_fn=ssl_prefix_collate_fn,
    num_workers=0,
)

In [ ]:
# verify one batch
batch = next(iter(ssl_loader))

print("sequence_lengths:", batch["sequence_lengths"])
print("time_steps shape:", batch["time_steps"].shape)
print("time_gaps shape:", batch["time_gaps"].shape)
print("padding_mask shape:", batch["padding_mask"].shape)
print("view_1_numerical shape:", batch["view_1_numerical"].shape)
print("view_1_histogram shape:", batch["view_1_histogram"].shape)
print("view_2_numerical shape:", batch["view_2_numerical"].shape)
print("view_2_histogram shape:", batch["view_2_histogram"].shape)

print(
    "Numerical views identical?",
    torch.allclose(batch["view_1_numerical"], batch["view_2_numerical"])
)
print(
    "Histogram views identical?",
    torch.allclose(batch["view_1_histogram"], batch["view_2_histogram"])
)

In [ ]:
# move tensors to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

batch_gpu = {
    k: v.to(device) if torch.is_tensor(v) else v
    for k, v in batch.items()
}

print(batch_gpu["view_1_numerical"].device)
print(batch_gpu["view_1_histogram"].device)

In [ ]:
batch = next(iter(ssl_loader))

device = torch.device("cuda")
batch = {k: v.to(device) if torch.is_tensor(v) else v for k, v in batch.items()}

print(batch["view_1_numerical"].device)
print(batch["view_1_histogram"].device)

In [ ]:
# NT-Xent loss implementation
import torch
import torch.nn.functional as F


def nt_xent_loss(
    z1: torch.Tensor,
    z2: torch.Tensor,
    temperature: float = 0.5,
) -> torch.Tensor:
    """
    Compute NT-Xent (SimCLR) contrastive loss.

    Parameters
    ----------
    z1, z2 : (B, D)
        Two augmented views of the same batch.
    temperature : float
        Scaling parameter.

    Returns
    -------
    torch.Tensor
        Scalar loss.
    """
    batch_size = z1.size(0)

    # Normalize
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    # Concatenate
    z = torch.cat([z1, z2], dim=0)  # (2B, D)

    # Cosine similarity matrix
    sim = torch.matmul(z, z.T)  # (2B, 2B)

    # Remove self-similarity
    mask = torch.eye(2 * batch_size, device=z.device).bool()
    sim = sim.masked_fill(mask, -1e9)

    # Scale by temperature
    sim = sim / temperature

    # Positive pairs:
    # (i, i+B) and (i+B, i)
    targets = torch.arange(batch_size, device=z.device)
    targets = torch.cat([targets + batch_size, targets], dim=0)

    loss = F.cross_entropy(sim, targets)

    return loss

In [ ]:
# new pretraining round
# batch diagnostics function

def compute_ssl_batch_diagnostics(
    pooled_1: torch.Tensor,
    pooled_2: torch.Tensor,
    proj_1: torch.Tensor,
    proj_2: torch.Tensor,
) -> dict[str, float]:
    """
    Compute batch-level SSL diagnostics.

    Parameters
    ----------
    pooled_1, pooled_2 : torch.Tensor
        Pooled encoder representations of shape (B, D_pooled).
    proj_1, proj_2 : torch.Tensor
        Projection-head outputs of shape (B, D_proj).

    Returns
    -------
    dict[str, float]
        Dictionary containing dispersion, norm, and similarity metrics.
    """
    with torch.no_grad():
        pooled_std = pooled_1.std(dim=0).mean().item()
        proj_std = proj_1.std(dim=0).mean().item()

        pooled_norm = pooled_1.norm(dim=1).mean().item()
        proj_norm = proj_1.norm(dim=1).mean().item()

        proj_1_n = F.normalize(proj_1, dim=1)
        proj_2_n = F.normalize(proj_2, dim=1)

        # Positive similarity: matching pairs
        pos_sim = (proj_1_n * proj_2_n).sum(dim=1).mean().item()

        # Negative similarity: average over non-matching cross-view pairs
        sim_matrix = torch.matmul(proj_1_n, proj_2_n.T)  # (B, B)
        batch_size = sim_matrix.size(0)
        neg_mask = ~torch.eye(batch_size, device=sim_matrix.device).bool()
        neg_sim = sim_matrix[neg_mask].mean().item()

    return {
        "pooled_std": pooled_std,
        "proj_std": proj_std,
        "pooled_norm": pooled_norm,
        "proj_norm": proj_norm,
        "pos_sim": pos_sim,
        "neg_sim": neg_sim,
    }

In [ ]:
# training loop with diagnostics
import numpy as np
import torch


def train_ssl_with_diagnostics(
    model: torch.nn.Module,
    train_loader,
    device: torch.device,
    num_epochs: int = 20,
    learning_rate: float = 1e-4,
    temperature: float = 0.5,
    weight_decay: float = 0.0,
):
    """
    Train SSL model with contrastive loss and representation diagnostics.

    Parameters
    ----------
    model : torch.nn.Module
        SSL sequence model.
    train_loader :
        DataLoader yielding SSL batches with two augmented views.
    device : torch.device
        CUDA or CPU device.
    num_epochs : int, default=10
        Number of training epochs.
    learning_rate : float, default=1e-4
        Optimizer learning rate.
    temperature : float, default=0.5
        NT-Xent temperature.
    weight_decay : float, default=0.0
        Adam weight decay.

    Returns
    -------
    dict
        History dictionary with tracked metrics.
    """
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = {
        "loss": [],
        "pooled_std": [],
        "proj_std": [],
        "pooled_norm": [],
        "proj_norm": [],
        "pos_sim": [],
        "neg_sim": [],
    }

    for epoch in range(num_epochs):
        model.train()

        total_loss = 0.0
        total_pooled_std = 0.0
        total_proj_std = 0.0
        total_pooled_norm = 0.0
        total_proj_norm = 0.0
        total_pos_sim = 0.0
        total_neg_sim = 0.0
        num_batches = 0

        for batch in train_loader:
            batch = {
                k: v.to(device) if torch.is_tensor(v) else v
                for k, v in batch.items()
            }

            pooled_1, proj_1 = model(
                numerical=batch["view_1_numerical"],
                histogram=batch["view_1_histogram"],
                time_gaps=batch["time_gaps"],
                padding_mask=batch["padding_mask"],
            )

            pooled_2, proj_2 = model(
                numerical=batch["view_2_numerical"],
                histogram=batch["view_2_histogram"],
                time_gaps=batch["time_gaps"],
                padding_mask=batch["padding_mask"],
            )

            loss = nt_xent_loss(
                z1=proj_1,
                z2=proj_2,
                temperature=temperature,
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            diagnostics = compute_ssl_batch_diagnostics(
                pooled_1=pooled_1,
                pooled_2=pooled_2,
                proj_1=proj_1,
                proj_2=proj_2,
            )

            total_loss += loss.item()
            total_pooled_std += diagnostics["pooled_std"]
            total_proj_std += diagnostics["proj_std"]
            total_pooled_norm += diagnostics["pooled_norm"]
            total_proj_norm += diagnostics["proj_norm"]
            total_pos_sim += diagnostics["pos_sim"]
            total_neg_sim += diagnostics["neg_sim"]
            num_batches += 1

        avg_loss = total_loss / max(num_batches, 1)
        avg_pooled_std = total_pooled_std / max(num_batches, 1)
        avg_proj_std = total_proj_std / max(num_batches, 1)
        avg_pooled_norm = total_pooled_norm / max(num_batches, 1)
        avg_proj_norm = total_proj_norm / max(num_batches, 1)
        avg_pos_sim = total_pos_sim / max(num_batches, 1)
        avg_neg_sim = total_neg_sim / max(num_batches, 1)

        history["loss"].append(avg_loss)
        history["pooled_std"].append(avg_pooled_std)
        history["proj_std"].append(avg_proj_std)
        history["pooled_norm"].append(avg_pooled_norm)
        history["proj_norm"].append(avg_proj_norm)
        history["pos_sim"].append(avg_pos_sim)
        history["neg_sim"].append(avg_neg_sim)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {avg_loss:.4f} | "
            f"Pooled std: {avg_pooled_std:.4f} | "
            f"Proj std: {avg_proj_std:.4f} | "
            f"Pooled norm: {avg_pooled_norm:.4f} | "
            f"Proj norm: {avg_proj_norm:.4f} | "
            f"Pos sim: {avg_pos_sim:.4f} | "
            f"Neg sim: {avg_neg_sim:.4f}"
        )

    return history

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
from ssl_model import SSLSequenceModel
ssl_model = SSLSequenceModel(
    numerical_dim=8,
    histogram_dim=97,
    numerical_hidden_dim=64,
    histogram_hidden_dim=64,
    projection_dim=64,
).to(device)

ssl_history = train_ssl_with_diagnostics(
    model=ssl_model,
    train_loader=ssl_loader,
    device=device,
    num_epochs=20,
    learning_rate=1e-4,
    temperature=0.5,
    weight_decay=0.0,
)

In [ ]:
# plotting history
import matplotlib.pyplot as plt


def plot_ssl_history(history: dict):
    metrics = [
        "loss",
        "pooled_std",
        "proj_std",
        "pooled_norm",
        "proj_norm",
        "pos_sim",
        "neg_sim",
    ]

    for metric in metrics:
        plt.figure()
        plt.plot(history[metric], marker="o")
        plt.title(metric)
        plt.xlabel("Epoch")
        plt.ylabel(metric)
        plt.grid(True)
        plt.show()

In [ ]:
plot_ssl_history(ssl_history)

In [ ]:
#Save the pretrained model
import torch

save_path = MODELS_DIR / "ssl_pretrained_simclr.pt"

torch.save({
    "model_state_dict": ssl_model.state_dict(),
    "numerical_dim": 8,
    "histogram_dim": 97,
    "numerical_hidden_dim": 64,
    "histogram_hidden_dim": 64,
    "projection_dim": 64,
    "history" : ssl_history,
    "num_epochs": 20,
    "notes": "mild augmentation setting",
}, save_path)

print("Model saved at:", save_path)